In [2]:
import cv2
import numpy as np
import pickle
import mediapipe as mp
from tensorflow.keras.models import load_model

model = load_model('motion_model.h5')
data = pickle.load(open('motion_data.pickle', 'rb'))
actions = data['actions']

mp_hands = mp.solutions.hands

hands = mp_hands.Hands(
    static_image_mode=False,
    min_detection_confidence=0.5,
    max_num_hands=1
)

def extract_features(hand_landmarks):
    x_, y_ = [], []
    for lm in hand_landmarks.landmark:
        x_.append(lm.x)
        y_.append(lm.y)

    features = []
    for lm in hand_landmarks.landmark:
        features.append(lm.x - min(x_))
        features.append(lm.y - min(y_))

    return features


sequence = []

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    frame = cv2.flip(frame, 1)

    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(img_rgb)

    if results.multi_hand_landmarks:
        hand_landmarks = results.multi_hand_landmarks[0]
        features = extract_features(hand_landmarks)
    else:
        features = [0]*42

    sequence.append(features)
    sequence = sequence[-30:]

    if len(sequence) == 30:
        prediction = model.predict(np.expand_dims(sequence, axis=0))[0]
        action = actions[np.argmax(prediction)]

        cv2.putText(frame, action, (50, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1,
                    (0, 255, 0), 2)

    cv2.imshow('Motion Recognition', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

1/1 [==============================] - 0s 25ms/step
